# Quadratic Hawkes (exp synapse): 1-loop rate correction vs simulation

`k=1` sibling of `pipeline_quad_expg_sim_compare.ipynb`.  Where the
cross-correlator notebook compares a τ-dependent C^(2)(τ), this one
compares the **firing rate** under three lenses:

* `sim`        — empirical mean rate from the numba simulator
* `tree`       — mean-field rate `n*` (FieldTheory saddle point)
* `tree+loop`  — `n*` + the 1-loop correction `δn*_loop` from
                 `compute_cumulants(k=1, max_ell=1)`

The 1-loop correction is what diagnoses whether the quadratic
nonlinearity in `phi(v) = a v²` is meaningfully shifting rates beyond
the mean-field saddle.  In the linear-phi case it would vanish.


## 1. Setup

In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, time
import numpy as np
import matplotlib.pyplot as plt

# Pipeline (theory side)
sys.path.insert(0, os.path.abspath('..'))
from pipeline import compute_cumulants
from pipeline.theories.quad_hawkes_2pop_expg import HAWKES_MODEL

# Sim side — only need the simulator, no k-point slice estimator
from models.hawkes_sim_quad_expg_numba import sim_hawkes_quad_expg_numba


## 2. Configuration

Same `fundamental` dict the cross-correlator notebook uses, but with
`k=1, max_ell=1` to pull the 1-loop rate correction out of the
pipeline.  No τ grid is needed — the rate is τ-independent — but we
keep a tiny one for the pipeline's internal evaluation API.


In [ ]:
fundamental = {
    'E':     [0.78, 0.81],
    'w':     [[0.3, 0.25], [0.3, 0.35]],
    'tau':   10.0,
    'a':     0.44,
    'tau_g': 2.5,
}

# k=1 (single-leg cumulant) at 1-loop level
k        = 1
max_ell  = 1
external_fields = [('dn', 1)]    # which population's rate to extract

# Tiny τ grid — k=1 stationary rates are τ-independent so a single
# point would do, but a 5-point grid lets us assert constancy as a
# sanity check.
tau_max  = 20.0
tau_step = 5.0

# Pipeline parallelism (same flag set as the k=2 notebook)
PARALLEL  = True
N_WORKERS = None

# Simulation
N_RUNS   = 5
T_sim    = float(2_000_000)     # total time per run
dt_sim   = 0.02                  # Euler step
dt_bin   = 0.25                  # binning resolution (only used to
                                 # match the simulator API; rate is
                                 # bin-resolution independent)

print(f'k={k}, max_ell={max_ell}, external_fields={external_fields}')
print(f'tau_max={tau_max}, tau_step={tau_step}')
print(f'PARALLEL={PARALLEL}, N_WORKERS={N_WORKERS}')
print(f'N_RUNS={N_RUNS}, T_sim={T_sim:.0g}, dt_sim={dt_sim}, dt_bin={dt_bin}')


## 3. Theory side — tree + 1-loop rate

`compute_cumulants(k=1, max_ell=1)` returns:

* `mf_values['nstar']` — the tree-level saddle-point rate `n*` per
  population
* `C_tau` — the 1-loop *correction* `δn*_loop` evaluated on the τ
  grid (constant for stationary systems; we average over τ to get
  the scalar)

The full 1-loop-corrected rate is `n*[i] + δn*_loop` for the
`external_fields[0]` population.


In [ ]:
t0 = time.perf_counter()
th = compute_cumulants(
    model           = HAWKES_MODEL,
    k               = k,
    max_ell         = max_ell,
    fundamental     = fundamental,
    external_fields = external_fields,
    tau_max         = tau_max,
    tau_step        = tau_step,
    parallel        = PARALLEL,
    n_workers       = N_WORKERS,
    verbose         = True,
)
print(f'\nTheory side took {time.perf_counter() - t0:.1f}s')

# Mean-field tree rate
nstar  = np.array(th['mf_values']['nstar'], dtype=float)
vstar  = np.array(th['mf_values']['vstar'], dtype=float)
print(f'\nMF tree rates n*:  {nstar}')
print(f'MF voltages v*:    {vstar}')

# 1-loop correction is τ-independent for stationary systems → average
C_arr  = th['C_tau']
ext_pop = external_fields[0][1] - 1   # 1-based → 0-based pop index
loop_correction_pop = float(C_arr.mean().real)

# Pad into a full per-population vector: only ext_pop is computed,
# others remain 0 (would require separate compute_cumulants calls).
loop_correction = np.zeros_like(nstar)
loop_correction[ext_pop] = loop_correction_pop

rate_tree     = nstar
rate_tree_loop = nstar + loop_correction

# Sanity: confirm the τ-grid result really is constant (stationarity)
spread = float(np.max(np.abs(C_arr - C_arr.mean()))) / max(abs(C_arr.mean()), 1e-12)
print(f'\n1-loop correction (pop {ext_pop+1}):  {loop_correction_pop:+.6e}')
print(f'  τ-grid relative spread:           {spread:.2e}  '
      f'(should be << 1 for stationary systems)')
print(f'\nTheory diagrams: {len(th["diagrams"])}')


## 4. Simulation side

Run `N_RUNS` independent simulations and collect the mean firing rate
per population.  No factorial-cumulant slice is needed at k=1 — the
rate is just `total_spikes / T_sim`.


In [ ]:
import secrets as _secrets

E_sim     = [float(x) for x in fundamental['E']]
w_sim     = [[float(x) for x in row] for row in fundamental['w']]
tau_sim   = float(fundamental['tau'])
tau_g_sim = float(fundamental['tau_g'])
a_sim     = float(fundamental['a'])
npop_sim  = len(E_sim)

n_steps        = int(T_sim / dt_sim)
bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
n_bins         = n_steps // bin_size_steps

v_init = np.array(vstar, dtype=float)
E_arr  = np.array(E_sim,  dtype=float)
W_arr  = np.array(w_sim,  dtype=float)

# JIT warmup
_ = sim_hawkes_quad_expg_numba(
    int(1000), float(dt_sim), float(tau_sim),
    float(tau_g_sim), float(a_sim),
    E_arr, W_arr, v_init.copy(),
    int(bin_size_steps), int(100), int(0),
)

BASE_SEED = _secrets.randbits(31)

rate_runs = []
t0 = time.perf_counter()
for run in range(N_RUNS):
    seed = int(BASE_SEED + run)
    binned_counts, voltage_bins, total_spikes = sim_hawkes_quad_expg_numba(
        int(n_steps), float(dt_sim), float(tau_sim),
        float(tau_g_sim), float(a_sim),
        E_arr, W_arr, v_init.copy(),
        int(bin_size_steps), int(n_bins), seed,
    )
    rate_runs.append(
        [float(total_spikes[i]) / T_sim for i in range(npop_sim)]
    )
    print(f'  run {run+1}/{N_RUNS}: rates = {rate_runs[-1]}')

rate_runs       = np.array(rate_runs)
rate_sim_mean   = rate_runs.mean(axis=0)
rate_sim_sem    = (rate_runs.std(axis=0, ddof=1) / np.sqrt(N_RUNS)
                   if N_RUNS > 1 else np.zeros_like(rate_sim_mean))

print(f'\nSimulation side took {time.perf_counter() - t0:.1f}s '
      f'({N_RUNS} runs × T={T_sim:.0g})')
print(f'  Sim mean rates ± SEM:')
for i in range(npop_sim):
    print(f'    pop {i+1}:  {rate_sim_mean[i]:.6f}  ± {rate_sim_sem[i]:.6f}')


## 5. Comparison — three rate estimates per population

Bar plot: simulation / mean-field / mean-field + 1-loop, with the sim
SEM as an error bar.  If the 1-loop correction is the right size, the
green (sim) bar should sit on top of the orange (tree+loop) bar within
the sim SEM.

Only the `external_fields[0]` population gets a non-zero `δn*_loop`
in this notebook — the other populations require separate
`compute_cumulants` calls (one per external leg).  Their orange bars
will match the blue (tree) bars exactly.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(npop_sim)
width = 0.27

# Tree (mean-field saddle)
ax.bar(x - width, rate_tree, width,
       label='Theory: tree (n*)', color='#3498DB', edgecolor='black')

# Tree + 1-loop
ax.bar(x, rate_tree_loop, width,
       label='Theory: tree + 1-loop', color='#F39C12', edgecolor='black')

# Simulation (with SEM error bars)
ax.bar(x + width, rate_sim_mean, width, yerr=rate_sim_sem,
       label='Simulation', color='#2ECC71', edgecolor='black',
       capsize=4)

ax.set_xticks(x)
ax.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
ax.set_ylabel('Firing rate')
ax.set_title(f'Tree vs 1-loop vs simulation '
             f'(quad Hawkes, T={T_sim:.0g}, '
             f'N_RUNS={N_RUNS})')
ax.legend()
ax.grid(True, axis='y', alpha=0.25)
fig.tight_layout()
plt.show()


## 6. Numerical residual

Per-population summary table.  The "residual / SEM" column is the
acid test: ≲1 means the 1-loop theory is consistent with the sim
within Monte-Carlo noise; ≫1 means there's a real discrepancy
(higher-loop missing, or a bug somewhere).


In [ ]:
print(f'{"pop":>4} {"sim ± SEM":>22} {"tree (n*)":>14} '
      f'{"tree+loop":>14} {"loop":>14} {"resid/SEM":>12}')
print('-' * 86)
for i in range(npop_sim):
    sem = rate_sim_sem[i]
    resid = rate_sim_mean[i] - rate_tree_loop[i]
    resid_over_sem = (abs(resid) / sem) if sem > 0 else float('nan')
    print(f'{i+1:>4d} '
          f'{rate_sim_mean[i]:>10.6f} ± {sem:.6f}  '
          f'{rate_tree[i]:>14.6f} '
          f'{rate_tree_loop[i]:>14.6f} '
          f'{loop_correction[i]:>+14.6e} '
          f'{resid_over_sem:>12.2f}')


## 7. (Optional) Save NPZ + PDF report

Saves the rate triple (sim / tree / tree+loop) to `pipeline_outputs/
quad_expg_loop_rate/` along with the multi-page PDF diagram report
from `pipeline.generate_report`.


In [ ]:
out_dir = '../pipeline_outputs/quad_expg_loop_rate'
os.makedirs(out_dir, exist_ok=True)

# NPZ schema mirrors pipeline_quad_expg_sim_compare for cross-notebook
# consistency.  At k=1 the cumulant IS the firing rate (no τ axis),
# so C_theory_* are length-npop vectors instead of τ-arrays:
#   C_theory_tree   = mean-field saddle rate n*  (per population)
#   C_theory_total  = n* + δn*_loop              (per population)
#   C_theory_loop   = δn*_loop                   (zero for populations
#                                                  not in external_fields)
npz_path = f'{out_dir}/quad_expg_k{k}_ell{max_ell}.npz'
np.savez(
    npz_path,
    C_theory_tree    = rate_tree,         # = nstar
    C_theory_total   = rate_tree_loop,    # = nstar + loop_correction
    C_theory_loop    = loop_correction,   # pure-loop part
    C_sim_mean       = rate_sim_mean,
    C_sim_sem        = rate_sim_sem,
    nstar            = nstar,
    vstar            = vstar,
    fundamental_keys = np.array(list(fundamental.keys())),
    k                = np.array([k], dtype=int),
    max_ell          = np.array([max_ell], dtype=int),
    ext_pop          = np.array([ext_pop], dtype=int),
)
print(f'Saved: {npz_path}')

from pipeline import generate_report
pdf_path = f'{out_dir}/quad_expg_k{k}_ell{max_ell}_report.pdf'
generate_report(
    model           = HAWKES_MODEL,
    k               = k,
    fundamental     = fundamental,
    external_fields = external_fields,
    output_pdf      = pdf_path,
    result          = th,
    verbose         = False,
)
print(f'Saved: {pdf_path}')
